In [1]:
import sys
!{sys.executable} -m pip install pandas



In [2]:
import pandas as pd
import numpy as np
import re
from dateutil.parser import parse
from datetime import datetime


In [3]:
customers = pd.read_csv('clean/customers_dirty.csv')
customers.head()

FileNotFoundError: [Errno 2] No such file or directory: 'clean/customers_dirty.csv'

# =============================================================================
# Initial Data Inspection
# =============================================================================

In [123]:

customers.shape

(538, 13)

In [124]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        535 non-null    float64
 1   first_name         535 non-null    str    
 2   last_name          535 non-null    str    
 3   email              507 non-null    str    
 4   phone              441 non-null    str    
 5   gender             444 non-null    str    
 6   birth_date         504 non-null    str    
 7   registration_date  535 non-null    str    
 8   country            535 non-null    str    
 9   city               535 non-null    str    
 10  address            528 non-null    str    
 11  is_active          451 non-null    str    
 12  notes              134 non-null    str    
dtypes: float64(1), str(12)
memory usage: 54.8 KB


In [125]:
customers.describe()

,customer_id
count,535.000000
mean,268.000000
std,154.585467
min,1.000000
25%,134.500000
50%,268.000000
75%,401.500000
max,535.000000


In [126]:
customers.isna().sum()

customer_id            3
first_name             3
last_name              3
email                 31
phone                 97
gender                94
birth_date            34
registration_date      3
country                3
city                   3
address               10
is_active             87
notes                404
dtype: int64

In [127]:
# Check for rows where 'customer_id' is NaN
customers[customers["customer_id"].isna()]

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes
188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
# Drop rows where all elements are NaN
customers.dropna(how="all", inplace=True)

# =============================================================================
# Clean Customer ID
# =============================================================================

In [129]:
# Check the distribution of values in each column, including NaN
for col in customers.columns:
    print(f"=== {col} ===")
    print(customers[col].value_counts(dropna=False))
    print()

=== customer_id ===
customer_id
185.0    1
526.0    1
40.0     1
114.0    1
138.0    1
        ..
333.0    1
270.0    1
338.0    1
92.0     1
81.0     1
Name: count, Length: 535, dtype: int64

=== first_name ===
first_name
Robert      11
Amy         10
Michael      9
David        8
John         8
            ..
Kristie      1
Melanie      1
Natalie      1
KIMBERLY     1
Dawn         1
Name: count, Length: 291, dtype: int64

=== last_name ===
last_name
Johnson     12
Smith       12
Anderson    10
Williams     7
Miller       6
            ..
Townsend     1
Wilson       1
Shaffer      1
Downs        1
Santiago     1
Name: count, Length: 341, dtype: int64

=== email ===
email
NaN                                28
                                   10
ariana.diaz@outlook.com             3
april.hernandez@fakemail.xyz        3
sara.mcdowell@yahoo.com             3
                                   ..
kimberly.wilson@yahoo.com           1
sarah.rivera@notarealdomain.zzz     1
edward.shaffer3

In [130]:
customers["customer_id"].isna().sum()

np.int64(0)

In [131]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [132]:
(customers["customer_id"] < 0).sum()

np.int64(0)

In [133]:
# Ensure customer IDs are stored as integers.
customers["customer_id"] = customers["customer_id"].astype(int)

In [134]:
# Check if all customer_id values are numeric
numeric_mask = pd.to_numeric(customers["customer_id"], errors="coerce").notna()
print("All customer_id values numeric?", numeric_mask.all())
print("Non-numeric customer_id count:", (~numeric_mask).sum())
customers.loc[~numeric_mask, "customer_id"]

All customer_id values numeric? True
Non-numeric customer_id count: 0


Series([], Name: customer_id, dtype: int64)

In [135]:
# check for outliers in customer_id using the IQR method
cid = pd.to_numeric(customers["customer_id"], errors="coerce")

print("Statistics on customer_id:")
print(cid.describe())

Q1 = cid.quantile(0.25)
Q3 = cid.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nQ1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Lower bound = {lower_bound}, Upper bound = {upper_bound}")

# create a boolean mask for outliers
outlier_mask = (cid < lower_bound) | (cid > upper_bound)
print("\nNumber of Outliers:", int(outlier_mask.sum()))

outliers = customers.loc[outlier_mask]
print("\nThe outliers are in the following rows:")
print(outliers)

# Add a new column to the DataFrame indicating whether each customer_id is an outlier
customers["customer_id_outlier"] = outlier_mask

# Display the min, max, median, and mean of customer_id
print("\nMin:", cid.min())
print("Max:", cid.max())
print("Median:", cid.median())
print("Mean:", cid.mean())

Statistics on customer_id:
count    535.000000
mean     268.000000
std      154.585467
min        1.000000
25%      134.500000
50%      268.000000
75%      401.500000
max      535.000000
Name: customer_id, dtype: float64

Q1 = 134.5, Q3 = 401.5, IQR = 267.0
Lower bound = -266.0, Upper bound = 802.0

Number of Outliers: 0

The outliers are in the following rows:
Empty DataFrame
Columns: [customer_id, first_name, last_name, email, phone, gender, birth_date, registration_date, country, city, address, is_active, notes]
Index: []

Min: 1
Max: 535
Median: 268.0
Mean: 268.0


# =============================================================================
# Clean First Name and Last Name
# ============================================================================= 

In [136]:
customers[["first_name", "last_name"]].head(20)

,first_name,last_name
0,Michael,Johnson
1,Daniel,Pacheco
2,Maurice,Barajas
3,Kirk,Morrison
4,Janet,Vaughn
5,Janet,Shepherd
6,Brittany,Burns
7,Kimberly,Franklin
8,Jacqueline,Moody
9,Eric,Baker


In [137]:
# Clean first_name and last_name columns
for col in ["first_name", "last_name"]:
    print(f"{col} NaNs before:", customers[col].isna().sum())
    # strip leading/trailing spaces, collapse multiple spaces, title case
    customers[col] = customers[col].str.strip()
    customers[col] = customers[col].str.replace(r"\s+", " ", regex=True)
    customers[col] = customers[col].str.title()
    print(f"{col} NaNs after :", customers[col].isna().sum())
    print(customers[col].value_counts().head(20))
    print()

first_name NaNs before: 0
first_name NaNs after : 0
first_name
Amy         11
Robert      11
Michael     10
John         9
Daniel       8
David        8
Michelle     8
Paul         7
Sarah        7
Jessica      7
Donna        7
Mary         7
Jennifer     6
Ashley       5
Richard      5
Steven       5
Charles      5
April        5
Kimberly     4
Heather      4
Name: count, dtype: int64

last_name NaNs before: 0
last_name NaNs after : 0
last_name
Smith        14
Johnson      13
Anderson     10
Thompson      8
Davis         7
Williams      7
Miller        7
Hernandez     6
Garcia        6
Jones         5
Young         5
Thomas        5
Brown         4
Rivera        4
Taylor        4
Moore         4
Morrison      3
Vaughn        3
Burns         3
Clark         3
Name: count, dtype: int64



In [138]:
customers["first_name"].isna().sum()

np.int64(0)

In [139]:
# Check for rows with leading/trailing spaces in first_name and last_name
for col in ["first_name", "last_name"]:
    print(f"Rows with leading/trailing spaces in {col}:")
    print(customers[customers[col] != customers[col].str.strip()])
    print()

Rows with leading/trailing spaces in first_name:
Empty DataFrame
Columns: [customer_id, first_name, last_name, email, phone, gender, birth_date, registration_date, country, city, address, is_active, notes, customer_id_outlier]
Index: []

Rows with leading/trailing spaces in last_name:
Empty DataFrame
Columns: [customer_id, first_name, last_name, email, phone, gender, birth_date, registration_date, country, city, address, is_active, notes, customer_id_outlier]
Index: []



In [140]:
# Collapse multiple spaces in first_name and last_name
for col in ["first_name", "last_name"]:
    customers[col] = customers[col].str.replace(r"\s+", " ", regex=True)

In [141]:
# Capitalize the first letter of each word in first_name and last_name
for col in ["first_name", "last_name"]:
    customers[col] = customers[col].str.title()

In [142]:


pattern = r"[^A-Za-zÀ-ÖØ-öø-ÿ' -]"

for col in ["first_name", "last_name"]:
    s = customers[col].astype(str)
    mask = s.str.contains(pattern, na=False)
    print(f"{col}: {mask.sum()} rows with special characters")
    special_chars = s[mask].str.findall(pattern).explode().dropna().unique()
    print("Unique special characters:", list(special_chars))
    if mask.any():
        print(customers.loc[mask, ["customer_id", "first_name", "last_name"]].head(50))
    print()

first_name: 0 rows with special characters
Unique special characters: []

last_name: 0 rows with special characters
Unique special characters: []



In [143]:
customers["first_name"].value_counts().head(20)

first_name
Amy         11
Robert      11
Michael     10
John         9
Daniel       8
David        8
Michelle     8
Paul         7
Sarah        7
Jessica      7
Donna        7
Mary         7
Jennifer     6
Ashley       5
Richard      5
Steven       5
Charles      5
April        5
Kimberly     4
Heather      4
Name: count, dtype: int64

In [144]:
customers["last_name"].value_counts().head(20)

last_name
Smith        14
Johnson      13
Anderson     10
Thompson      8
Davis         7
Williams      7
Miller        7
Hernandez     6
Garcia        6
Jones         5
Young         5
Thomas        5
Brown         4
Rivera        4
Taylor        4
Moore         4
Morrison      3
Vaughn        3
Burns         3
Clark         3
Name: count, dtype: int64

# =============================================================================
# Clean Email Addresses
# =============================================================================

In [145]:


# Validate email addresses using a regex pattern

email_pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

customers["email_valid"] = customers["email"].astype(str).str.match(email_pattern)

print("Valid emails:", customers["email_valid"].sum())
print("Invalid emails:", (~customers["email_valid"]).sum())

invalid_emails = customers[~customers["email_valid"]]
print("\nExamples of invalid emails:")
print(invalid_emails[["customer_id", "email"]].head(20))

Valid emails: 413
Invalid emails: 122

Examples of invalid emails:
    customer_id                              email
3           114                                   
5            58             janet.shepherd317@.com
15          180                   travis.johnston@
18          254          paul.martin@outlook.com  
22          518                                NaN
26          192                     shannon.brown@
27          170                                NaN
31          271             rebecca.newman771@.com
36          380         kevin.burns@fakemail.xyz  
41          394                  donna.chavez@.com
54          446                jason.williams@.com
57          118                                NaN
62          517         kayla.schwartz@gmail.com  
65          200                                NaN
67          278             ashley.millergmail.com
71           84                      jason.barnes@
73          396                                NaN
75          503

In [146]:

# Verify NaN count in email column
print(f"NaN values in email column: {customers['email'].isna().sum()}")

NaN values in email column: 28


In [147]:
# Clean email column by stripping leading/trailing spaces
customers["email"] = customers["email"].str.strip()

In [148]:
customers["email"] = customers["email"].str.lower()

In [149]:
customers[
    ~customers["email"].str.match(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        na=False
    )
]

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
3,114,Kirk,Morrison,,NaN,O,25-06-1967,06 Jan 2023,CA,Toronto,7059 Hill Turnpike Apt. 165,No,NaN,False,False
5,58,Janet,Shepherd,janet.shepherd317@.com,7762696538,male,06-23-64,05-17-25,Britain,Cairoo,0289 Kane Well,1,unknown,False,False
15,180,Travis,Johnston,travis.johnston@,,Male,13 Sep 1953,20231216,Egpyt,Huston,6519 Brittany Plains Apt. 807,Y,NaN,False,False
22,518,Amy,Morrison,NaN,NaN,FEMALE,09/24/1993,2025/04/12,EGYPT,houston,7212 Jill Crest,False,NaN,False,False
26,192,Shannon,Brown,shannon.brown@,+33 3 70 11 32 63,f,1962/06/13,2026/02/11,algeria,Boston,2368 Ortiz Overpass,Y,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,224,Lori,Thompson,lori.thompson190@,call me maybe,M,05/12/2006,2026-02-15,DZ,Houston,831 Garcia Underpass Apt. 100,Yes,Unknown,False,False
500,249,Todd,Lynch,todd.lynch208@.com,+1-593-876-4129,Male,NaN,30 Jul 2025,Canada,Paris,2996 Randy Isle Apt. 074,No,NaN,False,False
502,497,Michelle,Bradley,,901.768.7131,m,"February 16, 2003",14 Jun 2023,ALGERIA,seattle,79056 Morris Courts Suite 135,NaN,,False,False
521,327,Wendy,Chan,NaN,+1-887-655-3022,NaN,1962-01-02,16/06/2023,Spain,Berlinn,9380 Samantha Plaza Suite 895,NaN,NaN,False,False


In [150]:

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
invalid_mask = ~customers["email"].str.match(email_pattern, na=False)

In [151]:
customers.loc[invalid_mask, "email"] = np.nan

In [152]:
# check the number of invalid emails after cleaning
invalid_emails = customers[invalid_mask]
print(f"Number of invalid emails: {len(invalid_emails)}")

Number of invalid emails: 88


In [153]:
# validate email addresses again after cleaning
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

customers[
    customers["email"].notna() &
    ~customers["email"].str.match(email_pattern)
]

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid


In [154]:

customers[customers["email"].duplicated(keep=False)].sort_values("email")

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
471,304,Emily,Young,aaron.garrison@outlook.com,call me maybe,m,19561127,20250607,Algeria,Chicago,57759 Steele Drives,Y,NaN,False,True
374,74,Aaron,Garrison,aaron.garrison@outlook.com,,NaN,1946/05/16,07-04-23,﻿Egypt,TORONTO,421 David Knolls,Inactive,unknown,False,True
103,203,Aaron,Woods,aaron.woods@notarealdomain.zzz,+33 4 12 98 58 55,Female,22 Jun 1945,17-02-2026,GERMANY,Chigaco,700 Montoya Wall,Yes,NaN,False,True
211,398,Leslie,Fox,aaron.woods@notarealdomain.zzz,(830) 541-2939,f,02-23-10,14.12.2023,England,Seatle,99787 Morton Ridge Apt. 430,No,,False,True
138,511,Andrea,Grant,andrea.grant@yahoo.com,00213532551544,Female,1948-12-03,2026/04/15,Germny,New York,4956 Melissa Shoals Apt. 372,No,-,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,224,Lori,Thompson,NaN,call me maybe,M,05/12/2006,2026-02-15,DZ,Houston,831 Garcia Underpass Apt. 100,Yes,Unknown,False,False
500,249,Todd,Lynch,NaN,+1-593-876-4129,Male,NaN,30 Jul 2025,Canada,Paris,2996 Randy Isle Apt. 074,No,NaN,False,False
502,497,Michelle,Bradley,NaN,901.768.7131,m,"February 16, 2003",14 Jun 2023,ALGERIA,seattle,79056 Morris Courts Suite 135,NaN,,False,False
521,327,Wendy,Chan,NaN,+1-887-655-3022,NaN,1962-01-02,16/06/2023,Spain,Berlinn,9380 Samantha Plaza Suite 895,NaN,NaN,False,False


In [155]:
# Check for duplicate customers based on first_name, last_name, and email
customers[
    customers.duplicated(
        subset=["first_name", "last_name", "email"],
        keep=False
    )
].sort_values(["first_name", "last_name"])

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
138,511,Andrea,Grant,andrea.grant@yahoo.com,00213532551544,Female,1948-12-03,2026/04/15,Germny,New York,4956 Melissa Shoals Apt. 372,No,-,False,True
212,323,Andrea,Grant,andrea.grant@yahoo.com,00213532551544,Female,1948-12-03,2026/04/15,Germny,New York,4956 Melissa Shoals Apt. 372,No,-,False,True
209,535,April,Hernandez,april.hernandez@fakemail.xyz,+33 4 77 82 68 15,M,"July 04, 2002",20241128,germany,Paris,9526 Wheeler Parkways,False,,False,True
491,132,April,Hernandez,april.hernandez@fakemail.xyz,+1-222-351-3320,M,"July 04, 2002",20241128,germany,Paris,9526 Wheeler Parkways,False,,False,True
182,479,April,Watson,april.watson@company.co,NaN,Unknown,21/11/1971,31 Mar 2024,Spain,LONDON,NaN,Inactive,NaN,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,504,Sydney,Taylor,sydney.taylor@notarealdomain.zzz,(898) 527-6692,FEMALE,01/04/2008,05.10.2023,Germany,Duabi,516 Joseph Crossing Suite 549,True,,False,True
429,513,Tanner,Bird,tanner.bird550@hotmail.com,00213544880838,NaN,19450402,17/05/2025,Emirates,BERLIN,46786 Morrow Cliffs,0,NaN,False,True
447,120,Tanner,Bird,tanner.bird550@hotmail.com,00213544880838,NaN,19450402,17/05/2025,Emirates,BERLIN,46786 Morrow Cliffs,0,NaN,False,True
65,200,Troy,Morris,NaN,273-9899,Unknown,15-10-1956,"June 14, 2023",U.S.A.,Pariss,750 Cory Tunnel,True,NaN,False,False


In [156]:
customers[["customer_id", "email"]].sample(30, random_state=44)

,customer_id,email
275,388,amanda.terrell@outlook.com
259,384,NaN
93,378,NaN
484,161,kevin.stout914@company.co
214,248,eddie.matthews@yahoo.com
466,265,jessica.carlson@fakemail.xyz
238,104,stacey.walker434@yahoo.com
203,159,meredith.thomas@mail.ru
521,327,NaN
352,287,NaN


In [157]:
#i wanted to use this function to keep only the top-level domain (TLD) in the email addresses, but there more domains than just the common ones, so I commented it out. If you want to use it, you can uncomment and modify the valid_tlds set as needed.


# #valid_tlds = {
#     "com", "net", "org", "edu", "gov", "mil", "info", "biz",
#     "co", "io", "us", "uk", "ca", "de", "fr", "es", "it", "nl",
#     "ru", "br", "au", "jp", "in", "za", "mx", "ae", "sa", "se",
#     "no", "ch", "pl", "kr", "tr", "gr"
# }
# 
# def keep_tld_only(email):
#     if not isinstance(email, str):
#         return np.nan
#     email = email.strip()
#     if email == "":
#         return np.nan
#     match = re.match(r"^([^@\s]+)@([^@\s]+)$", email)
#     if not match:
#         return np.nan
#     local_part, domain = match.groups()
#     if "." not in domain:
#         return np.nan
#     tld = domain.rsplit(".", 1)[1].lower()
#     if tld in valid_tlds:
#         return f"{local_part}@.{tld}"
#     return np.nan
# 
# customers["email"] = customers["email"].apply(keep_tld_only)
# 
# print("Emails kept:", customers["email"].notna().sum())
# print("Emails deleted:", customers["email"].isna().sum())
# customers[["customer_id", "email"]].head(20)

In [158]:

# all in one code for cleaning and validating email addresses

email_pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

def clean_email(email):

    # Missing values
    if pd.isna(email):
        return np.nan

    # Convert to string
    email = str(email)

    # Remove spaces
    email = email.strip()

    # Empty string -> NaN
    if email == "":
        return np.nan

    # Lowercase
    email = email.lower()

    # Validate format
    if not re.match(email_pattern, email):
        return np.nan

    # Keep valid email
    return email

# Apply cleaning
customers["email"] = customers["email"].apply(clean_email)

In [159]:
customers[["customer_id", "email"]].sample(30, random_state=44)

,customer_id,email
275,388,amanda.terrell@outlook.com
259,384,NaN
93,378,NaN
484,161,kevin.stout914@company.co
214,248,eddie.matthews@yahoo.com
466,265,jessica.carlson@fakemail.xyz
238,104,stacey.walker434@yahoo.com
203,159,meredith.thomas@mail.ru
521,327,NaN
352,287,NaN


In [160]:
# Check the distribution of top-level domains (TLDs) in the email addresses 
customers["email"].str.extract(r"\.([A-Za-z]+)$")[0].value_counts(dropna=False)

0
com    253
NaN     88
xyz     57
zzz     55
co      41
ru      41
Name: count, dtype: int64

In [161]:
# Check the distribution of email domains in the email addresses
customers["email"].str.extract(r"@([^\.]+)")[0].value_counts(dropna=False)

0
NaN               88
test              61
fakemail          57
yahoo             56
notarealdomain    55
outlook           49
gmail             45
hotmail           42
company           41
mail              41
Name: count, dtype: int64

In [162]:
# Check the distribution of email domains in the email addresses
customers["email"].str.extract(r"@(.+)")[0].value_counts(dropna=False)

0
NaN                   88
test.com              61
fakemail.xyz          57
yahoo.com             56
notarealdomain.zzz    55
outlook.com           49
gmail.com             45
hotmail.com           42
company.co            41
mail.ru               41
Name: count, dtype: int64

# =============================================================================
# Standardize Country Names
# =============================================================================

In [163]:
# Check the distribution of countries in the customers dataset
customers["country"].value_counts(dropna=False)

country
Egypt       23
Algeria     16
UK          16
Spain       16
Usa         16
            ..
G﻿ermny      1
S​pain       1
 Franc       1
Ger many     1
England      1
Name: count, Length: 61, dtype: int64

In [164]:
sorted(customers["country"].dropna().unique())

['ALGERIA',
 'Algeria',
 'Algerie',
 'Alge\u200bria',
 'Britain',
 'CA',
 'CANADA',
 'Canada',
 'Canad\ufeffa',
 'Canda',
 'DZ',
 'Deutschland',
 'EGYPT',
 'Egpyt',
 'Egypt',
 'Egyp\ufefft',
 'Eg\u200bpyt',
 'Emirates',
 'England',
 'England\xa0',
 'Espana',
 'FRANCE',
 'Fr',
 'Franc',
 'France',
 'GERMANY',
 'Germany',
 'Germny',
 'Ger\xa0many',
 'G\ufeffermny',
 'SPAIN',
 'Spain',
 'Spain ',
 'Spa\ufeffin ',
 'S\u200bpain ',
 'U.A.E.',
 'U.K.',
 'U.S.A.',
 'UAE',
 'UK',
 'US',
 'USA',
 'United Arab Emirates',
 'United Kingdom',
 'United States',
 'United States of America',
 'Usa',
 'U\u200b.A.E.',
 'algeria',
 'canada',
 'egypt',
 'france',
 'germany',
 'spain',
 'united kingdom',
 'united states',
 '\xa0Egypt',
 '\xa0Franc',
 '\xa0egypt',
 '\ufeffAlgerie',
 '\ufeffEgypt']

In [165]:
# Check for leading/trailing spaces in 'country'
customers["country"].str.contains(r"^\s|\s$", regex=True, na=False).sum()

np.int64(15)

In [166]:
#remove special characters from 'country' column
customers["country"] = customers["country"].str.replace(r"[^A-Za-zÀ-ÖØ-öø-ÿ' -]", "", regex=True)

In [167]:
# Check for leading/trailing spaces in 'country' after removing special characters
customers["country"] = (
    customers["country"]
    .str.strip()          
    .str.lower()          
)

In [168]:
sorted(customers["country"].dropna().unique())

['algeria',
 'algerie',
 'britain',
 'ca',
 'canada',
 'canda',
 'deutschland',
 'dz',
 'egpyt',
 'egypt',
 'emirates',
 'england',
 'espana',
 'fr',
 'franc',
 'france',
 'germany',
 'germny',
 'spain',
 'uae',
 'uk',
 'united arab emirates',
 'united kingdom',
 'united states',
 'united states of america',
 'us',
 'usa']

In [169]:
# Check the distribution of countries in the customers dataset
customers["country"].value_counts(dropna=False)

country
spain                       50
egypt                       48
germany                     42
algeria                     39
usa                         38
canada                      38
france                      28
uk                          24
uae                         21
united kingdom              20
united states               15
canda                       15
emirates                    14
deutschland                 14
egpyt                       13
espana                      13
germny                      13
ca                          12
britain                     12
england                     11
us                          10
united states of america     9
united arab emirates         8
fr                           8
algerie                      8
dz                           7
franc                        5
Name: count, dtype: int64

In [170]:
#if you want to use fuzzy matching to clean the 'country' column, you can use the following code. Make sure to install the `rapidfuzz` library if you haven't already.

# from rapidfuzz import process
# 
# countries = [
#     "Algeria",
#     "Germany",
#     "France",
#     "Spain",
#     "Egypt",
#     "Canada",
#     "United States",
#     "United Kingdom",
#     "United Arab Emirates"
# ]
# 
# def clean_country(country):
#     if pd.isna(country):
#         return np.nan
# 
#     match = process.extractOne(country, countries)
# 
#     if match and match[1] >= 85:   # seuil de confiance
#         return match[0]
# 
#     return country

In [171]:
country_dict = {
    # Algeria
    "algeria": "Algeria",
    "algerie": "Algeria",
    "dz": "Algeria",

    # Egypt
    "egypt": "Egypt",
    "egpyt": "Egypt",

    # Germany
    "germany": "Germany",
    "germny": "Germany",
    "deutschland": "Germany",

    # Spain
    "spain": "Spain",
    "espana": "Spain",

    # France
    "france": "France",
    "fr": "France",
    "franc": "France",

    # Canada
    "canada": "Canada",
    "canda": "Canada",
    "ca": "Canada",

    # United Kingdom
    "uk": "United Kingdom",
    "united kingdom": "United Kingdom",
    "england": "United Kingdom",
    "britain": "United Kingdom",

    # United States
    "usa": "United States",
    "us": "United States",
    "united states": "United States",
    "united states of america": "United States",

    # United Arab Emirates
    "uae": "United Arab Emirates",
    "emirates": "United Arab Emirates",
    "united arab emirates": "United Arab Emirates",
}

In [172]:
customers["country"] = customers["country"].replace(country_dict)

In [173]:
customers["country"].value_counts(dropna=False)

country
United States           72
Germany                 69
United Kingdom          67
Canada                  65
Spain                   63
Egypt                   61
Algeria                 54
United Arab Emirates    43
France                  41
Name: count, dtype: int64

In [174]:
customers["country"].unique()

<StringArray>
[             'Germany',                'Egypt',              'Algeria',
               'Canada',        'United States',       'United Kingdom',
 'United Arab Emirates',                'Spain',               'France']
Length: 9, dtype: str

In [175]:
customers["country"].isna().sum()

np.int64(0)

# =============================================================================
# Clean Phone Numbers
# =============================================================================

In [176]:
customers["phone"].value_counts(dropna=False)

phone
NaN                  94
call me maybe        50
                     35
+1-541-208-2682       2
(389) 715-8811        2
                     ..
+1-578-762-1870       1
(400) 680-3038        1
+213 713485913        1
+33 2 57 75 83 12     1
6025070544            1
Name: count, Length: 347, dtype: int64

In [177]:
customers["phone"].sample(30, random_state=42)

117         900.507.5073
132                     
154       +213 732573855
246       +213 528566572
84        (898) 527-6692
347                  NaN
6         (467) 509-6091
479                  NaN
182                  NaN
55            3884119695
104           7706119814
533       +213 713485913
321         869.451.7278
250        call me maybe
238                  NaN
470      +1-978-936-8231
526           3277102223
358           9506222658
212       00213532551544
496           6209579572
10        (329) 944-3369
406                  NaN
148       +213 514307663
452       (275) 716-7805
2                       
418       00213530793878
505    +33 7 36 61 33 10
185      +1-370-813-5491
414         618.349.1471
196       (738) 757-9312
Name: phone, dtype: str

In [178]:
# Check for phone numbers containing letters
customers.loc[
    customers["phone"].str.contains(r"[A-Za-z]", na=False),
    "phone"
] = np.nan

In [179]:
customers["phone"].value_counts(dropna=False)

phone
NaN                  144
                      35
+1-541-208-2682        2
(389) 715-8811         2
+1-547-257-6533        2
                    ... 
+1-578-762-1870        1
(400) 680-3038         1
+213 713485913         1
+33 2 57 75 83 12      1
6025070544             1
Name: count, Length: 346, dtype: int64

In [180]:
# Clean phone numbers by removing non-digit characters
customers["phone"] = (
    customers["phone"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

In [181]:
customers["phone"].sample(30, random_state=42)

117        9005075073
132                  
154      213732573855
246      213528566572
84         8985276692
347               NaN
6          4675096091
479               NaN
182               NaN
55         3884119695
104        7706119814
533      213713485913
321        8694517278
250               NaN
238               NaN
470       19789368231
526        3277102223
358        9506222658
212    00213532551544
496        6209579572
10         3299443369
406               NaN
148      213514307663
452        2757167805
2                    
418    00213530793878
505       33736613310
185       13708135491
414        6183491471
196        7387579312
Name: phone, dtype: str

In [182]:
customers["phone"].str.len().value_counts(dropna=False).sort_index()

phone
0.0      35
7.0      40
10.0    135
11.0     89
12.0     49
14.0     43
NaN     144
Name: count, dtype: int64

In [183]:
# Check for phone numbers with length 7 and set them to NaN
customers.loc[
    customers["phone"].str.len() == 7,
    "phone"
] = pd.NA

In [184]:
# Check for phone numbers with length 0 and set them to NaN
customers.loc[
    customers["phone"].str.len() == 0,
    "phone"
] = pd.NA

In [185]:
# Check for phone numbers with length 14 and display the country and phone columns
customers.loc[
    customers["phone"].str.len() == 14,
    ["country", "phone"]
].head(20)

,country,phone
39,France,00213790524312
52,United Kingdom,00213748573324
67,Spain,00213762713153
72,Egypt,00213776772528
86,Canada,00213670289903
87,Spain,00213691686468
97,Egypt,00213698116346
128,United States,00213602588544
131,Spain,00213769382988
134,Egypt,00213675738226


In [186]:
# Remove leading "00" from phone numbers
customers["phone"] = customers["phone"].str.replace(r"^00", "", regex=True)

In [187]:
customers["phone"].str.len().value_counts(dropna=False).sort_index()

phone
10.0    135
11.0     89
12.0     92
NaN     219
Name: count, dtype: int64

In [188]:
# Check for phone numbers that consist of the same digit repeated multiple times
customers[
    customers["phone"].str.fullmatch(r"(\d)\1+")
]

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid


In [189]:
# Check for phone numbers that consist of the same digit repeated multiple times
test_numbers = [
    "1234567890",
    "0123456789",
    "0987654321",
]

customers[customers["phone"].isin(test_numbers)]

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid


# =============================================================================
# Standardize City Names
# =============================================================================

In [190]:

sorted(customers["city"].dropna().str.strip().str.lower().unique())

['algeirs',
 'alger',
 'algiers',
 'algies',
 'berlin',
 'berlinn',
 'bosotn',
 'boston',
 'cairo',
 'cairoo',
 'chicago',
 'chigaco',
 'duabi',
 'dubai',
 'houston',
 'huston',
 'kairo',
 'l.a.',
 'la',
 'london',
 'londonn',
 'londres',
 'los angelas',
 'los angeles',
 'madird',
 'madrid',
 'maimi',
 'miami',
 'new  york',
 'new york',
 'newyork',
 'ney york',
 'paris',
 'pariss',
 'seatle',
 'seattle',
 'toronto',
 'torronto']

In [191]:
#remove leading/trailing spaces and title case in city column
customers["city"] = customers["city"].str.strip().str.title()

In [192]:
customers["city"] = customers["city"].str.lower()

In [193]:
#remove special characters from 'city' column
customers["city"] = customers["city"].str.replace(r"[^A-Za-zÀ-ÖØ-öø-ÿ' -]", "", regex=True)

In [194]:
customers.groupby("city").size()

city
algeirs         5
alger           4
algiers        15
algies          7
berlin         30
berlinn         2
bosotn          2
boston         25
cairo          18
cairoo          9
chicago        22
chigaco        21
duabi           5
dubai          26
houston        37
huston          5
kairo           6
la             12
london         18
londonn         5
londres         4
los angelas     7
los angeles    18
madird         12
madrid         37
maimi          14
miami          22
new  york       7
new york       20
newyork         2
ney york        1
paris          28
pariss          9
seatle          6
seattle        28
toronto        40
torronto        6
dtype: int64

In [195]:
city_dict = {
    # Algiers
    "algiers": "Algiers",
    "algeirs": "Algiers",
    "algies": "Algiers",
    "alger": "Algiers",

    # Berlin
    "berlin": "Berlin",
    "berlinn": "Berlin",

    # Boston
    "boston": "Boston",
    "bosotn": "Boston",

    # Cairo
    "cairo": "Cairo",
    "cairoo": "Cairo",
    "kairo": "Cairo",

    # Chicago
    "chicago": "Chicago",
    "chigaco": "Chicago",

    # Dubai
    "dubai": "Dubai",
    "duabi": "Dubai",

    # Houston
    "houston": "Houston",
    "huston": "Houston",

    # London
    "london": "London",
    "londonn": "London",
    "londres": "London",

    # Los Angeles
    "la": "Los Angeles",
    "los angeles": "Los Angeles",
    "los angelas": "Los Angeles",

    # Madrid
    "madrid": "Madrid",
    "madird": "Madrid",

    # Miami
    "miami": "Miami",
    "maimi": "Miami",

    # New York
    "new york": "New York",
    "newyork": "New York",
    "new  york": "New York",
    "ney york": "New York",

    # Paris
    "paris": "Paris",
    "pariss": "Paris",

    # Seattle
    "seattle": "Seattle",
    "seatle": "Seattle",

    # Toronto
    "toronto": "Toronto",
    "torronto": "Toronto"
}

In [196]:
customers["city"] = (
    customers["city"]
    .str.strip()
    .str.lower()
    .replace(city_dict)
)

In [197]:
customers["city"].value_counts().sort_index()

city
Algiers        31
Berlin         32
Boston         27
Cairo          33
Chicago        43
Dubai          31
Houston        42
London         27
Los Angeles    37
Madrid         49
Miami          36
New York       30
Paris          37
Seattle        34
Toronto        46
Name: count, dtype: int64

# =============================================================================
# Standardize Gender Values
# =============================================================================

In [198]:
customers["gender"].value_counts(dropna=False)

gender
NaN        91
FEMALE     37
F          37
Male       36
male       36
MALE       36
Unknown    36
Other      36
female     35
Female     31
f          28
m          27
M          26
X          22
O          21
Name: count, dtype: int64

In [199]:
# transform gender column to lowercase
customers["gender"] = customers["gender"].str.lower()

In [200]:
customers["gender"].value_counts(dropna=False)

gender
male       108
female     103
NaN         91
f           65
m           53
unknown     36
other       36
x           22
o           21
Name: count, dtype: int64

In [201]:
gender_dict = {
    "m": "male",
    "f": "female",
    "o": "other",
    "x": "other"
}

customers["gender"] = customers["gender"].replace(gender_dict)

In [202]:
customers["gender"].value_counts(dropna=False)

gender
female     168
male       161
NaN         91
other       79
unknown     36
Name: count, dtype: int64

# =============================================================================
# Standardize active status values.
# =============================================================================

In [203]:
customers["is_active"].value_counts(dropna=False)

is_active
NaN         84
No          56
Y           54
True        49
False       46
0           45
Inactive    45
Yes         43
1           39
Active      39
N           35
Name: count, dtype: int64

In [204]:
# convert is_active column to lowercase and strip leading/trailing spaces
customers["is_active"] = (
    customers["is_active"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [205]:
customers["is_active"].value_counts(dropna=False)

is_active
<NA>        84
no          56
y           54
true        49
false       46
0           45
inactive    45
yes         43
1           39
active      39
n           35
Name: count, dtype: Int64

In [206]:
status_dict = {
    "yes": "active",
    "y": "active",
    "true": "active",
    "1": "active",

    "no": "inactive",
    "n": "inactive",
    "false": "inactive",
    "0": "inactive"
}

customers["is_active"] = customers["is_active"].replace(status_dict)

In [207]:
customers["is_active"].value_counts(dropna=False)

is_active
inactive    227
active      224
<NA>         84
Name: count, dtype: Int64

# =============================================================================
# Clean Notes
# =============================================================================

In [208]:
customers["notes"].value_counts(dropna=False)

notes
NaN                                       401
                                           27
-                                          17
                                           17
unknown                                    14
Unknown                                    14
Memory number occur behind leave.           2
Reveal laugh attack and.                    1
Speak close join improve put.               1
✅ Listen car.                               1
Wear talk between rate.***                  1
Federal right firm movie decade.            1
Any factor police baby. 🚀                   1
These three spring.                         1
Politics few each.                          1
Parent its approach.                        1
Small think law air.                        1
Relate center book money half.              1
Here control affect task.                   1
Likely machine.                             1
Factor as.                                  1
People drive tree central.  

In [209]:
customers["notes"] = (
    customers["notes"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [210]:
customers["notes"] = customers["notes"].replace({
    "": pd.NA,
    "-": pd.NA,
    "unknown": pd.NA,
    "n/a": pd.NA,
    "null": pd.NA
})

In [211]:
customers["notes"].value_counts(dropna=False)

notes
<NA>                                      490
memory number occur behind leave.           2
reveal laugh attack and.                    1
speak close join improve put.               1
✅ listen car.                               1
wear talk between rate.***                  1
federal right firm movie decade.            1
any factor police baby. 🚀                   1
these three spring.                         1
politics few each.                          1
parent its approach.                        1
small think law air.                        1
relate center book money half.              1
here control affect task.                   1
likely machine.                             1
factor as.                                  1
people drive tree central.                  1
head garden.                                1
lot notice just particularly single. ❌      1
get every visit right.                      1
response official. 🙄                        1
impact fly bit claim.       

In [212]:
#Remove special characters from 'notes' column
customers["notes"] = customers["notes"].str.replace(r'[^\w\s]', '', regex=True)

In [213]:
customers["notes"].value_counts(dropna=False)

notes
<NA>                                    490
memory number occur behind leave          2
reveal laugh attack and                   1
speak close join improve put              1
 listen car                               1
wear talk between rate                    1
federal right firm movie decade           1
any factor police baby                    1
these three spring                        1
politics few each                         1
parent its approach                       1
small think law air                       1
relate center book money half             1
here control affect task                  1
likely machine                            1
factor as                                 1
people drive tree central                 1
head garden                               1
lot notice just particularly single       1
get every visit right                     1
response official                         1
impact fly bit claim                      1
in miss                   

# =============================================================================
# Clean Birth Dates
# =============================================================================

In [214]:
customers["birth_date"].value_counts(dropna=False)

birth_date
NaN                  31
0000-00-00           14
31/02/2023           12
13/13/2022           11
27-07-1936            2
                     ..
February 05, 1995     1
19410529              1
1948/10/06            1
October 05, 1959      1
05/01/1966            1
Name: count, Length: 440, dtype: int64

In [215]:
customers["birth_date"].sample(30, random_state=42)

117        05-19-99
132      03/10/1943
154      31/02/2023
246    May 15, 1953
84       01/04/2008
347        08-11-58
6        30-03-1940
479      07-09-1984
182      21/11/1971
55       08.09.1942
104        06-12-52
533        19410529
321      11/08/1950
250        19580721
238    May 10, 1969
470      1959-03-11
526      21.07.1946
358      2008-08-05
212      1948-12-03
496      29-06-1952
10       27-07-1936
406      05/09/1947
148      1973/11/19
452      14.11.1992
2        19-10-1976
418        19890804
505      23.09.1955
185             NaN
414     15 Aug 1977
196      02.09.1954
Name: birth_date, dtype: str

In [216]:
# birth_date cleaning and validation using dateutil.parser and pandas

from dateutil.parser import parse
from datetime import datetime
import pandas as pd

# =============================================================================
# Birth Date Cleaning
# =============================================================================

INVALID_DATES = {
    "",
    "unknown",
    "n/a",
    "na",
    "null",
    "none",
    "0000-00-00"
}

CURRENT_YEAR = datetime.today().year
MIN_YEAR = 1915

def clean_birth_date(date):
    """
    Clean messy birth dates and convert them to datetime.

    Rules:
    - Missing or invalid values -> NaT
    - Supports multiple date formats
    - Invalid calendar dates -> NaT
    - Birth years before 1900 or after the current year -> NaT
    """

    if pd.isna(date):
        return pd.NaT

    date = str(date).strip()

    if date.lower() in INVALID_DATES:
        return pd.NaT

    try:
        parsed_date = parse(date, fuzzy=False)

        if parsed_date.year < MIN_YEAR or parsed_date.year > CURRENT_YEAR:
            return pd.NaT

        return pd.Timestamp(parsed_date)

    except (ValueError, TypeError):
        return pd.NaT


# Apply cleaning
customers["birth_date"] = customers["birth_date"].apply(clean_birth_date)

# =============================================================================
# Summary
# =============================================================================

print("Birth Date Cleaning Summary")
print("-" * 30)
print(f"Total Rows    : {len(customers)}")
print(f"Valid Dates   : {customers['birth_date'].notna().sum()}")
print(f"Invalid Dates : {customers['birth_date'].isna().sum()}")

print("\nData Type:")
print(customers["birth_date"].dtype)

print("\nSample:")
print(customers["birth_date"].sample(10, random_state=42))

Birth Date Cleaning Summary
------------------------------
Total Rows    : 535
Valid Dates   : 415
Invalid Dates : 120

Data Type:
datetime64[us]

Sample:
117   1999-05-19
132   1943-03-10
154          NaT
246   1953-05-15
84    2008-01-04
347          NaT
6     1940-03-30
479   1984-07-09
182   1971-11-21
55    1942-08-09
Name: birth_date, dtype: datetime64[us]


In [217]:
customers.head(10)

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
0,185,Michael,Johnson,michael.johnson@test.com,NaN,male,NaT,06-13-23,Germany,Los Angeles,12545 Mueller Pike Suite 571,<NA>,reveal laugh attack and,False,True
1,526,Daniel,Pacheco,daniel.pacheco@fakemail.xyz,18139027050,NaN,NaT,07/24/2023,Egypt,Algiers,31190 Harper Squares,inactive,<NA>,False,True
2,40,Maurice,Barajas,maurice.barajas@gmail.com,NaN,other,1976-10-19,21-03-2025,Algeria,Madrid,744 Hall Stravenue Apt. 518,active,<NA>,False,True
3,114,Kirk,Morrison,NaN,NaN,other,1967-06-25,06 Jan 2023,Canada,Toronto,7059 Hill Turnpike Apt. 165,inactive,<NA>,False,False
4,138,Janet,Vaughn,janet.vaughn276@hotmail.com,33220955479,other,1972-03-12,06-19-25,United States,Chicago,8441 Huang Locks Suite 240,inactive,<NA>,False,True
5,58,Janet,Shepherd,NaN,7762696538,male,NaT,05-17-25,United Kingdom,Cairo,0289 Kane Well,active,<NA>,False,False
6,441,Brittany,Burns,brittany.burns@test.com,4675096091,male,1940-03-30,2024/12/27,United States,Algiers,8700 Brown Turnpike Apt. 613,inactive,<NA>,False,True
7,387,Kimberly,Franklin,kimberly.franklin270@outlook.com,NaN,female,1997-05-06,05/25/2024,United Arab Emirates,Madrid,9998 Abigail Mission Suite 038,inactive,<NA>,False,True
8,356,Jacqueline,Moody,jacqueline.moody@company.co,6369428652,male,1944-12-15,28 Apr 2026,United Kingdom,Houston,8573 Hannah Knolls,inactive,<NA>,False,True
9,367,Eric,Baker,eric.baker@fakemail.xyz,9586361288,male,2002-03-30,"October 06, 2023",Egypt,Houston,44196 Jessica Stream,active,<NA>,False,True


# =============================================================================
# Clean Registration Dates
# =============================================================================

In [218]:
customers["registration_date"].sample(50, random_state=1)

365         30 Jun 2024
428          05-11-2024
256            20260625
307          07.05.2023
301     August 27, 2024
533          02/25/2025
41          29 Dec 2024
427    October 19, 2024
388          03/17/2026
471            20250607
90           03/11/2024
34      August 17, 2024
378         21 Apr 2023
375          17-04-2023
65        June 14, 2023
9      October 06, 2023
432          07.04.2026
239          2023-12-22
507          26.12.2023
377          12/06/2023
251         14 Apr 2024
168          2025/02/05
526       June 17, 2025
179            02-14-25
356            20250403
306          2025-02-03
261          22/03/2024
416          19.02.2026
237          03-05-2026
132          04.06.2023
406         27 Jul 2025
465          16-08-2023
527            20240929
396            20231215
62           12/12/2024
119          11/30/2023
165          2023-04-26
285         22 Apr 2025
117     August 18, 2024
247          25-11-2024
324            07-01-26
308          13-

In [219]:


from dateutil.parser import parse
from datetime import datetime
import pandas as pd

# =============================================================================
# Registration Date Cleaning
# =============================================================================

INVALID_DATES = {
    "",
    "unknown",
    "n/a",
    "na",
    "null",
    "none",
    "0000-00-00"
}

CURRENT_DATE = datetime.today()
MIN_YEAR = 2000

def clean_registration_date(date):
    """
    Clean messy registration dates and convert them to datetime.

    Rules:
    - Missing or invalid values -> NaT
    - Supports multiple date formats
    - Invalid calendar dates -> NaT
    - Registration dates before 2000 or after today -> NaT
    """

    if pd.isna(date):
        return pd.NaT

    date = str(date).strip()

    if date.lower() in INVALID_DATES:
        return pd.NaT

    try:
        parsed_date = parse(date, fuzzy=False)

        if parsed_date.year < MIN_YEAR or parsed_date > CURRENT_DATE:
            return pd.NaT

        return pd.Timestamp(parsed_date)

    except (ValueError, TypeError):
        return pd.NaT


# Apply cleaning
customers["registration_date"] = customers["registration_date"].apply(clean_registration_date)

# =============================================================================
# Summary
# =============================================================================

print("Registration Date Cleaning Summary")
print("-" * 40)
print(f"Total Rows    : {len(customers)}")
print(f"Valid Dates   : {customers['registration_date'].notna().sum()}")
print(f"Invalid Dates : {customers['registration_date'].isna().sum()}")

print("\nData Type:")
print(customers["registration_date"].dtype)

print("\nSample:")
print(customers["registration_date"].sample(10, random_state=42))

Registration Date Cleaning Summary
----------------------------------------
Total Rows    : 535
Valid Dates   : 534
Invalid Dates : 1

Data Type:
datetime64[us]

Sample:
117   2024-08-18
132   2023-04-06
154   2023-06-21
246   2026-02-23
84    2023-05-10
347   2025-08-30
6     2024-12-27
479   2025-08-06
182   2024-03-31
55    2024-03-10
Name: registration_date, dtype: datetime64[us]


# =============================================================================
# Clean Addresses
# =============================================================================

In [220]:
customers["address"].value_counts(dropna=False).head(30)

address
NaN                                7
31190 Harper Squares               2
342 Teresa Courts                  2
7212 Jill Crest                    2
818 Nicholas Ford                  2
145 Jeffrey Dale Suite 279         2
76591 Jerry Ways                   2
4781 Debbie Common Suite 283       2
2140 Price Circle Suite 427        2
4713 Kevin Summit Suite 546        2
750 Cory Tunnel                    2
183 Patrick Spurs Suite 198        2
02499 Gonzales Mountain            2
516 Joseph Crossing Suite 549      2
28309 Guerrero Springs Apt. 074    2
1016 Evan Haven Apt. 795           2
25355 Page Rapids                  2
4956 Melissa Shoals Apt. 372       2
4170 Zoe Keys Apt. 100             2
83028 Young Street Suite 995       2
8755 Gibson Islands                2
  9557 Ellis Port Apt. 424         2
696 Michael Vista Suite 924        2
34125 William Manor Apt. 966       2
9526 Wheeler Parkways              2
82754 Christopher Park Apt. 647    2
082 Smith Forges Apt. 504     

In [221]:
customers["address"].describe()

count                      528
unique                     494
top       31190 Harper Squares
freq                         2
Name: address, dtype: object

In [222]:
# Clean address column by stripping leading/trailing spaces
customers["address"] = (
    customers["address"]
    .astype("string")
    .str.strip()
)

In [223]:
customers["address"] = customers["address"].replace("", pd.NA)

In [224]:
customers["address"].isna().sum()

np.int64(7)

In [225]:
# Check for leading/trailing spaces in all string columns and clean them
string_cols = customers.select_dtypes(include=["object", "string"]).columns

customers[string_cols] = customers[string_cols].apply(
    lambda s: s.str.replace(r"\s+", " ", regex=True).str.strip()
)

In [226]:
print("Address Cleaning Summary")
print("-" * 30)
print(f"Missing Addresses : {customers['address'].isna().sum()}")
print(f"Unique Addresses  : {customers['address'].nunique()}")

Address Cleaning Summary
------------------------------
Missing Addresses : 7
Unique Addresses  : 494


# =============================================================================
# Final Data Quality Check
# =============================================================================

In [227]:
# convert all columns to their best possible dtypes
customers = customers.convert_dtypes()

In [228]:
# Check for empty strings in the DataFrame and replace them with NaN
customers = customers.replace("", pd.NA)

In [229]:
customers.info()

<class 'pandas.DataFrame'>
Index: 535 entries, 0 to 537
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          535 non-null    Int64         
 1   first_name           535 non-null    string        
 2   last_name            535 non-null    string        
 3   email                447 non-null    string        
 4   phone                316 non-null    string        
 5   gender               444 non-null    string        
 6   birth_date           415 non-null    datetime64[us]
 7   registration_date    534 non-null    datetime64[us]
 8   country              535 non-null    string        
 9   city                 535 non-null    string        
 10  address              528 non-null    string        
 11  is_active            451 non-null    string        
 12  notes                45 non-null     string        
 13  customer_id_outlier  535 non-null    boolean       

In [230]:
# Check for missing values in the DataFrame
customers.isna().sum().sort_values(ascending=False)

notes                  490
phone                  219
birth_date             120
gender                  91
email                   88
is_active               84
address                  7
registration_date        1
first_name               0
last_name                0
customer_id              0
city                     0
country                  0
customer_id_outlier      0
email_valid              0
dtype: int64

In [231]:
customers.duplicated().sum()

np.int64(0)

In [232]:
duplicate_summary = customers.apply(lambda col: pd.Series({
    "duplicate_count": col.loc[col.notna()].duplicated(keep=False).sum(),
    "duplicate_value_count": col.loc[col.notna() & col.duplicated(keep=False)].nunique()
})).T

duplicate_summary.sort_values(by="duplicate_count", ascending=False)


,duplicate_count,duplicate_value_count
customer_id_outlier,535,1
city,535,15
country,535,9
email_valid,535,2
is_active,451,2
gender,444,4
first_name,406,123
last_name,338,116
registration_date,238,107
email,91,44


# Note:
Repeated names, cities, countries, and other categorical values are normal
and do not indicate duplicate customer records.
Further investigation should focus on duplicate email addresses,
phone numbers, and addresses.
Customer IDs are unique across the dataset.

In [233]:
customers["birth_date"].describe()

count                           415
mean     1972-05-08 03:52:28.915662
min             1935-08-16 00:00:00
25%             1953-02-20 00:00:00
50%             1972-03-12 00:00:00
75%             1991-09-21 00:00:00
max             2010-11-20 00:00:00
Name: birth_date, dtype: object

In [234]:
customers["email"].str.len().describe()

count        447.0
mean     25.552573
std       3.783524
min           17.0
25%           23.0
50%           25.0
75%           28.0
max           39.0
Name: email, dtype: Float64

In [235]:
customers["registration_date"].describe()

count                           534
mean     2024-09-30 04:02:41.797752
min             2023-01-01 00:00:00
25%             2023-12-01 06:00:00
50%             2024-10-04 00:00:00
75%             2025-07-07 00:00:00
max             2026-07-10 00:00:00
Name: registration_date, dtype: object

In [236]:
customers.sample(15, random_state=42)

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
117,302,Jason,Whitehead,ariana.diaz@outlook.com,9005075073,female,1999-05-19,2024-08-18,Egypt,Cairo,23943 Howard Union,active,<NA>,False,True
132,175,Sean,Juarez,sean.juarez@test.com,<NA>,male,1943-03-10,2023-04-06,United Kingdom,Algiers,67888 Garcia Corners,active,<NA>,False,True
154,324,Craig,Romero,craig.romero@hotmail.com,213732573855,male,NaT,2023-06-21,Spain,Algiers,60127 Davis Greens,active,<NA>,False,True
246,3,Michele,Jones,michele.jones@test.com,213528566572,<NA>,1953-05-15,2026-02-23,France,Chicago,82997 Kristen Valley Suite 311,active,<NA>,False,True
84,68,Sydney,Taylor,sydney.taylor@notarealdomain.zzz,8985276692,female,2008-01-04,2023-05-10,Germany,Dubai,516 Joseph Crossing Suite 549,active,<NA>,False,True
347,346,Jeremy,Davis,jeremy.davis@outlook.com,<NA>,male,NaT,2025-08-30,Algeria,Chicago,2400 Maurice Mountains Apt. 774,inactive,<NA>,False,True
6,441,Brittany,Burns,brittany.burns@test.com,4675096091,male,1940-03-30,2024-12-27,United States,Algiers,8700 Brown Turnpike Apt. 613,inactive,<NA>,False,True
479,91,Carrie,Wolf,carrie.wolf@company.co,<NA>,unknown,1984-07-09,2025-08-06,Algeria,Houston,779 Maldonado Port,inactive,<NA>,False,True
182,479,April,Watson,april.watson@company.co,<NA>,unknown,1971-11-21,2024-03-31,Spain,London,<NA>,inactive,<NA>,False,True
55,343,Jason,Sanders,jason.sanders@test.com,3884119695,<NA>,1942-08-09,2024-03-10,Canada,Dubai,006 Dalton River Apt. 164,active,<NA>,False,True


In [237]:
customers.describe(include="all")

,customer_id,first_name,last_name,email,phone,gender,birth_date,registration_date,country,city,address,is_active,notes,customer_id_outlier,email_valid
count,535.0,535,535,447,316,444,415,534,535,535,528,451,45,535,535
unique,<NA>,252,313,400,304,4,NaN,NaN,9,15,494,2,44,1,2
top,<NA>,Amy,Smith,ariana.diaz@outlook.com,15412082682,female,NaN,NaN,United States,Madrid,31190 Harper Squares,inactive,memory number occur behind leave,False,True
freq,<NA>,11,14,3,2,168,NaN,NaN,72,49,2,227,2,535,413
mean,268.0,NaN,NaN,NaN,NaN,NaN,1972-05-08 03:52:28.915662,2024-09-30 04:02:41.797752,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.0,NaN,NaN,NaN,NaN,NaN,1935-08-16 00:00:00,2023-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,134.5,NaN,NaN,NaN,NaN,NaN,1953-02-20 00:00:00,2023-12-01 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,268.0,NaN,NaN,NaN,NaN,NaN,1972-03-12 00:00:00,2024-10-04 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,401.5,NaN,NaN,NaN,NaN,NaN,1991-09-21 00:00:00,2025-07-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,535.0,NaN,NaN,NaN,NaN,NaN,2010-11-20 00:00:00,2026-07-10 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# =============================================================================
# Export Cleaned Dataset
# =============================================================================

In [238]:
# Export the cleaned customer dataset to a CSV file.
customers.to_csv("customers_clean.csv", index=False)
# Check the exported dataset.
print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.
